### Consolidated Environment Setup, Ollama Exposure, and Ngrok Tunneling

In [3]:
import platform
import subprocess
import torch
import os
import requests
import json
import time
import sys

# Ensure pyngrok is installed
try:
    import pyngrok
except ImportError:
    print("pyngrok not found, installing...")
    !pip install pyngrok -q
    import pyngrok

from pyngrok import ngrok, conf
from google.colab import userdata

# --- Global Variable Initializations (mimicking the notebook state) ---
globals()['local_model_tag'] = 'N/A'
globals()['local_inference_pass'] = False
globals()['ngrok_tags_external_pass'] = False
globals()['external_tags_status'] = 'N/A'
globals()['external_tags_headers'] = {}
globals()['external_tags_body'] = 'N/A'
globals()['external_tags_log_snippet'] = 'N/A'
globals()['ngrok_generate_external_pass'] = False
globals()['external_generate_status'] = 'N/A'
globals()['external_generate_headers'] = {}
globals()['external_generate_body'] = 'N/A'
globals()['external_generate_log_snippet'] = 'N/A'
globals()['final_ngrok_tunnel_active'] = False
globals()['ngrok_tunnel_url_final_test'] = 'N/A'
globals()['final_ngrok_forwarding_target'] = 'N/A'
globals()['local_ollama_reachable'] = False

# Define Ollama local endpoint
OLLAMA_LOCAL_ENDPOINT = "http://localhost:11434"
OLLAMA_PORT = 11434
ollama_log_path = 'ollama.log'

# Function to get new log entries since a given size
def get_new_log_entries(start_size):
    try:
        with open(ollama_log_path, 'r') as f:
            f.seek(start_size)
            return f.read()
    except Exception as e:
        return f"Error reading ollama.log: {e}"

def check_runtime_and_gpu():
    print("--- Step 1: Checking Runtime and GPU ---")
    try:
        print(f"Python Version: {platform.python_version()}")
        print(f"OS: {platform.system()} {platform.release()}")

        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
            vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
            print(f"GPU: {gpu_name}")
            print(f"VRAM: {vram:.2f} GB")
        else:
            print("GPU: No GPU detected. Please enable a GPU runtime.")

        print("Disk Space Available:")
        df_command = """df -h / | grep "/" | awk '{print "Disk Space Available: " $4}'"""
        result = subprocess.run(df_command, shell=True, check=True, capture_output=True, text=True)
        print(result.stdout.strip())

        print("Step 1: Runtime and GPU check successful.\n")
        return True
    except Exception as e:
        print(f"Step 1 Failed: {e}\n")
        return False

def install_ollama():
    print("--- Step 2: Installing Ollama ---")
    try:
        print("Updating apt-get...")
        subprocess.run(['sudo', 'apt-get', 'update'], check=True, capture_output=True, text=True)
        print("Installing zstd...")
        subprocess.run(['sudo', 'apt-get', 'install', '-y', 'zstd'], check=True, capture_output=True, text=True)
        print("Downloading and installing Ollama...")
        ollama_install_cmd = 'curl -fsSL https://ollama.com/install.sh | sh'
        subprocess.run(ollama_install_cmd, shell=True, check=True, capture_output=True, text=True)
        print("Step 2: Ollama installation successful.\n")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Step 2 Failed: Ollama installation process failed. Command: {e.cmd}")
        print(f"Stdout:\n{e.stdout}")
        print(f"Stderr:\n{e.stderr}")
        return False
    except Exception as e:
        print(f"Step 2 Failed: An unexpected error occurred during Ollama installation: {e}\n")
        return False

def start_ollama_server():
    print("--- Step 3: Starting Ollama Server ---")
    try:
        subprocess.run(['killall', '-q', 'ollama'], capture_output=True)
        time.sleep(1)

        os.system('nohup ollama serve > ollama.log 2>&1 &')
        print("Waiting for Ollama server to respond...")
        for i in range(20):
            try:
                resp = requests.get('http://localhost:11434', timeout=5)
                if resp.status_code == 200:
                    print("Ollama server is active at http://localhost:11434")
                    print("Step 3: Ollama server started successfully.\n")
                    globals()['local_ollama_reachable'] = True
                    return True
            except requests.exceptions.ConnectionError:
                pass
            except Exception as req_e:
                print(f"Warning during server check: {req_e}")
            time.sleep(2)
        print("Step 3 Failed: Timeout - Ollama server failed to start. Check ollama.log for details.\n")
        globals()['local_ollama_reachable'] = False
        return False
    except Exception as e:
        print(f"Step 3 Failed: {e}\n")
        return False

def pull_and_verify_models():
    print("--- Step 4: Pulling and Verifying Models ---")
    try:
        # Pull qwen3:8b
        print("Pulling qwen3:8b model...")
        pull_qwen3_8b_cmd = ['ollama', 'pull', 'qwen3:8b']
        process_qwen3_8b = subprocess.run(pull_qwen3_8b_cmd, check=True, capture_output=True, text=True, timeout=900)
        print(process_qwen3_8b.stdout)

        # Pull qwen2.5-coder:7b
        print("Pulling qwen2.5-coder:7b model...")
        pull_qwen2_5_coder_cmd = ['ollama', 'pull', 'qwen2.5-coder:7b']
        process_qwen2_5_coder = subprocess.run(pull_qwen2_5_coder_cmd, check=True, capture_output=True, text=True, timeout=900)
        print(process_qwen2_5_coder.stdout)

        # Pull qwen3.5:latest
        print("Pulling qwen3.5:latest model...")
        pull_qwen3_5_latest_cmd = ['ollama', 'pull', 'qwen3.5:latest']
        process_qwen3_5_latest = subprocess.run(pull_qwen3_5_latest_cmd, check=True, capture_output=True, text=True, timeout=900)
        print(process_qwen3_5_latest.stdout)


        resp = requests.get("http://localhost:11434/api/tags")
        resp.raise_for_status()
        models_data = resp.json()
        print("Local API Response:", json.dumps(models_data, indent=2))

        # Verify all three models
        qwen3_8b_found = False
        qwen2_5_coder_7b_found = False
        qwen3_5_latest_found = False

        for model in models_data.get('models', []):
            if model.get('name') == 'qwen3:8b':
                qwen3_8b_found = True
            if model.get('name') == 'qwen2.5-coder:7b':
                qwen2_5_coder_7b_found = True
            if model.get('name') == 'qwen3.5:latest':
                qwen3_5_latest_found = True

        if not (qwen3_8b_found and qwen2_5_coder_7b_found and qwen3_5_latest_found):
            print("Step 4 Failed: One or more models not found in Ollama API.\n")
            return False

        # Set local_model_tag to qwen3.5:latest as it's the last one requested and likely used in inference.
        globals()['local_model_tag'] = 'qwen3.5:latest'

        print("Step 4: Models pulled and verified successfully.\n")
        return True
    except requests.exceptions.RequestException as e:
        print(f"Step 4 Failed: API Error during model verification or pull: {e}\n")
        return False
    except subprocess.TimeoutExpired:
        print("Step 4 Failed: Timeout while pulling models. They might be too large or your internet connection is slow.\n")
        return False
    except subprocess.CalledProcessError as e:
        print(f"Step 4 Failed: Error pulling models. Stderr: {e.stderr}\n")
        return False
    except Exception as e:
        print(f"Step 4 Failed: {e}\n")
        return False

def run_inference_test():
    print("--- Step 5: Running Inference Test with qwen3.5 ---")
    payload = {
        "model": globals()['local_model_tag'],
        "prompt": "Provide a brief explanation of what AEGIS development might involve.",
        "stream": False,
        "options": {
            "num_predict": 100,
            "temperature": 0.7
        }
    }
    try:
        response = requests.post("http://localhost:11434/api/generate", json=payload, timeout=600)
        response.raise_for_status()
        result = response.json()
        print("Inference Success!")
        print("Response:", result.get('response'))
        print(f"Model used: {result.get('model')}")
        print("Step 5: Inference test successful.\n")
        globals()['local_inference_pass'] = True
        return True
    except requests.exceptions.RequestException as e:
        print(f"Step 5 Failed: Inference failed or request error: {e}\n")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response content: {e.response.text}")
        globals()['local_inference_pass'] = False
        return False
    except Exception as e:
        print(f"Step 5 Failed: {e}\n")
        globals()['local_inference_pass'] = False
        return False

def run_final_ngrok_test():
    print("--- Step 6: Final Ngrok Diagnostic Test with Host-Header Rewrite ---")
    # Terminate any running localtunnel process from previous attempts
    if 'lt_process' in globals() and globals()['lt_process'] is not None:
        print("Terminating existing localtunnel process...")
        globals()['lt_process'].terminate()
        globals()['lt_process'].wait()
        globals()['lt_process'] = None
        print("Localtunnel process terminated.")

    # Disconnect any existing ngrok tunnels to ensure a clean start
    print("Killing any existing ngrok tunnels to ensure a clean start...")
    try:
        ngrok.kill()
        time.sleep(1)
        print("Existing ngrok tunnels killed.")
    except Exception as e:
        print(f"No active ngrok tunnels to kill or error during kill: {e}")

    # Re-authenticate ngrok using the stored token
    ngrok_auth_token = userdata.get("NGROK_AUTH_TOKEN")
    if ngrok_auth_token is None:
        sys.exit("NGROK_AUTH_TOKEN not found. Aborting final tunnel setup.")
    else:
        ngrok.set_auth_token(ngrok_auth_token)
        print("ngrok authenticated for final test.")

    print(f"--- Starting ngrok tunnel for local port {OLLAMA_PORT} with host-header rewrite ---\n")

    try:
        tunnel_host_header = ngrok.connect(OLLAMA_PORT, "http", host_header=f"localhost:{OLLAMA_PORT}")
        ngrok_tunnel_url_final_test = tunnel_host_header.public_url

        print(f"Ngrok tunnel with host-header rewrite active at: {ngrok_tunnel_url_final_test}")
        globals()['ngrok_tunnel_url_final_test'] = ngrok_tunnel_url_final_test
        globals()['final_ngrok_tunnel_active'] = True
        globals()['final_ngrok_forwarding_target'] = f"localhost:{OLLAMA_PORT}"
        print(f"Ngrok forwarding target: {globals()['final_ngrok_forwarding_target']}")

    except Exception as e:
        print(f"Failed to create ngrok tunnel with host-header: {e}")
        globals()['final_ngrok_tunnel_active'] = False
        sys.exit("Ngrok tunnel creation failed for final test. Aborting.")

    print("\n--- Local Ollama Tests ---")

    # 1. Local: GET http://localhost:11434/api/tags
    print("\nTesting Local GET /api/tags...")
    try:
        local_tags_response = requests.get(f"{OLLAMA_LOCAL_ENDPOINT}/api/tags", timeout=10)
        local_tags_response.raise_for_status()
        local_tags_data = local_tags_response.json()
        print("Local GET /api/tags: PASS")
        print(f"Status: {local_tags_response.status_code}")
        print("Response (first 200 chars):", json.dumps(local_tags_data, indent=2)[:200], "...")
        globals()['local_ollama_reachable'] = True
        globals()['local_model_tag'] = 'qwen3.5:latest'
    except requests.exceptions.RequestException as e:
        print(f"Local GET /api/tags: FAIL - {e}")
        globals()['local_ollama_reachable'] = False
        globals()['local_model_tag'] = 'N/A'

    # 2. Local: POST http://localhost:11434/api/generate
    print("\nTesting Local POST /api/generate...")
    model_tag_local = globals()['local_model_tag'] # Use the tag confirmed above
    local_payload = {
        "model": model_tag_local,
        "prompt": "What is the capital of France?",
        "stream": False,
        "options": {
            "num_predict": 20,
            "temperature": 0.1
        }
    }
    try:
        local_generate_response = requests.post(f"{OLLAMA_LOCAL_ENDPOINT}/api/generate", json=local_payload, timeout=60)
        local_generate_response.raise_for_status()
        local_generate_data = local_generate_response.json()
        print("Local POST /api/generate: PASS")
        print(f"Status: {local_generate_response.status_code}")
        print("Response (first 100 chars):", local_generate_data.get('response', '')[:100], "...")
        globals()['local_inference_pass'] = True
    except requests.exceptions.RequestException as e:
        print(f"Local POST /api/generate: FAIL - {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response content: {e.response.text}")
        globals()['local_inference_pass'] = False

    print("\n--- External Ngrok Tests with Host-Header Rewrite ---")

    if not globals().get('final_ngrok_tunnel_active'):
        sys.exit("Ngrok tunnel not active. Aborting external tests.")

    ngrok_url = globals()['ngrok_tunnel_url_final_test']
    model_tag_external = globals()['local_model_tag'] # Use the tag confirmed above
    headers = {"Content-Type": "application/json"}

    initial_ollama_log_size = os.path.getsize(ollama_log_path)

    # 3. External: GET <ngrok-url>/api/tags
    print("\nTesting External GET /api/tags...")

    try:
        log_size_before_tags = os.path.getsize(ollama_log_path)
        external_tags_response = requests.get(f"{ngrok_url}/api/tags", headers=headers, timeout=30)
        globals()['external_tags_status'] = external_tags_response.status_code
        globals()['external_tags_headers'] = dict(external_tags_response.headers)
        globals()['external_tags_body'] = external_tags_response.text
        external_tags_response.raise_for_status()
        print("External GET /api/tags: PASS (Tunnel likely reached Ollama)")
        globals()['ngrok_tags_external_pass'] = True
    except requests.exceptions.RequestException as e:
        print(f"External GET /api/tags: FAIL - {e}")
        if hasattr(e, 'response') and e.response is not None:
            globals()['external_tags_status'] = e.response.status_code
            globals()['external_tags_headers'] = dict(e.response.headers)
            globals()['external_tags_body'] = e.response.text
        globals()['ngrok_tags_external_pass'] = False
    except Exception as e:
        print(f"External GET /api/tags: Unexpected error - {e}")
        globals()['ngrok_tags_external_pass'] = False
    finally:
        globals()['external_tags_log_snippet'] = get_new_log_entries(log_size_before_tags)

    # 4. External: POST <ngrok-url>/api/generate
    print("\nTesting External POST /api/generate...")

    external_payload = {
        "model": model_tag_external,
        "prompt": "Explain quantum entanglement in two sentences.",
        "stream": False,
        "options": {
            "num_predict": 50,
            "temperature": 0.1
        }
    }

    try:
        log_size_before_generate = os.path.getsize(ollama_log_path)
        external_generate_response = requests.post(f"{ngrok_url}/api/generate", json=external_payload, headers=headers, timeout=90)
        globals()['external_generate_status'] = external_generate_response.status_code
        globals()['external_generate_headers'] = dict(external_generate_response.headers)
        globals()['external_generate_body'] = external_generate_response.text
        external_generate_response.raise_for_status()
        print("External POST /api/generate: PASS (Tunnel likely reached Ollama)")
        globals()['ngrok_generate_external_pass'] = True
    except requests.exceptions.RequestException as e:
        print(f"External POST /api/generate: FAIL - {e}")
        if hasattr(e, 'response') and e.response is not None:
            globals()['external_generate_status'] = e.response.status_code
            globals()['external_generate_headers'] = dict(e.response.headers)
            globals()['external_generate_body'] = e.response.text
        globals()['ngrok_generate_external_pass'] = False
    except Exception as e:
        print(f"External POST /api/generate: Unexpected error - {e}")
        globals()['ngrok_generate_external_pass'] = False
    finally:
        globals()['external_generate_log_snippet'] = get_new_log_entries(log_size_before_generate)


def main_consolidated():
    if not check_runtime_and_gpu():
        sys.exit("Environment setup aborted.")

    if not install_ollama():
        sys.exit("Environment setup aborted.")

    if not start_ollama_server():
        sys.exit("Environment setup aborted.")

    if not pull_and_verify_models():
        sys.exit("Environment setup aborted.")

    if not run_inference_test():
        sys.exit("Environment setup aborted.")

    run_final_ngrok_test()

    print("\n--- Final Ngrok Host-Header Test Report ---")

    print(f"\n**LOCAL OLLAMA:**")
    print(f"- Ollama endpoint: {OLLAMA_LOCAL_ENDPOINT}")
    print(f"- model tag: {globals().get('local_model_tag', 'N/A')}")
    print(f"- local inference result: {'PASS' if globals().get('local_inference_pass', False) else 'FAIL'}")

    print(f"\n**NGROK (Host-Header Rewrite Test):**")
    print(f"- tunnel started: {'YES' if globals().get('final_ngrok_tunnel_active', False) else 'NO'}")
    print(f"- temporary URL: {globals().get('ngrok_tunnel_url_final_test', 'N/A')}")
    print(f"- ngrok forwarding target: {globals().get('final_ngrok_forwarding_target', 'N/A')}")

    print(f"\n- **/api/tags (External GET):** {'PASS' if globals().get('ngrok_tags_external_pass', False) else 'FAIL'}")
    print(f"  - HTTP status: {globals().get('external_tags_status', 'N/A')}")
    print(f"  - Response headers:\n```json\n{json.dumps(globals().get('external_tags_headers', {}), indent=2)}\n```")
    print(f"  - Response body:\n```\n{globals().get('external_tags_body', 'N/A')}\n```")
    print(f"  - Ollama server logs during request (snippet):\n```\n{globals().get('external_tags_log_snippet', 'N/A')}\n```")
    print(f"  - Request reached Ollama: {'YES' if 'http/1.1' in globals().get('external_tags_log_snippet', '').lower() else 'NO'}")

    print(f"\n- **/api/generate (External POST):** {'PASS' if globals().get('ngrok_generate_external_pass', False) else 'FAIL'}")
    print(f"  - HTTP status: {globals().get('external_generate_status', 'N/A')}")
    print(f"  - Response headers:\n```json\n{json.dumps(globals().get('external_generate_headers', {}), indent=2)}\n```")
    print(f"  - Response body:\n```\n{globals().get('external_generate_body', 'N/A')}\n```")
    print(f"  - Ollama server logs during request (snippet):\n```\n{globals().get('external_generate_log_snippet', 'N/A')}\n```")
    print(f"  - Request reached Ollama: {'YES' if 'http/1.1' in globals().get('external_generate_log_snippet', '').lower() else 'NO'}")

    print("\n---\n")

    if globals().get('external_tags_status') == 403 or globals().get('external_generate_status') == 403:
        print("\n--- Conclusion: HTTP 403 Forbidden persists even with host-header rewrite. ---")
        print("\nThe evidence suggests that the Colab environment has a fundamental restriction on exposing local services externally, regardless of the tunneling service (ngrok, localtunnel) or specific ngrok configurations like host-header rewriting. This is likely due to network policies or firewalls enforced by Google Colab.")
        print("Further attempts to troubleshoot tunneling are unlikely to succeed within this environment.")
    else:
        print("\n--- Conclusion: Ngrok host-header rewrite successfully exposed Ollama. ---")
        print("\nThe previous 403 errors were likely due to Ollama rejecting requests with an unrecognized Host header. The `host-header=\"localhost:11434\"` option resolved this issue.")
        print("\n--- Minimal Python requests example for an external client ---")
        ngrok_url = globals()['ngrok_tunnel_url_final_test']
        model_tag_final_example = globals()['local_model_tag'] # Use the tag confirmed above
        python_example_final = f"""
import requests
import json

OLLAMA_EXTERNAL_URL = "{ngrok_url}"

payload = {{
    "model": "{model_tag_final_example}",
    "prompt": "Tell me a fun fact about large language models.",
    "stream": False,
    "options": {{
        "num_predict": 50,
        "temperature": 0.1
    }}
}}

try:
    headers = {{\"Content-Type\": \"application/json\"}}
    response = requests.post(f"{{OLLAMA_EXTERNAL_URL}}/api/generate", data=json.dumps(payload), headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors
    result = response.json()
    print(\"External inference successful!\")
    print(\"Response:\", result.get('response'))
    print(f\"Model used: {{result.get('model')}}\")
except requests.exceptions.RequestException as e:
    print(f\"Request Error: {{e}}\")
    if hasattr(e, 'response') and e.response is not None:
        print(f\"Response status code: {{e.response.status_code}}\")
        print(f\"Response content: {{e.response.text}}\")
"""
        print(python_example_final)

    print("\n--- End of Diagnostic Experiment ---")

# Run the consolidated main function
if __name__ == "__main__":
    main_consolidated()

--- Step 1: Checking Runtime and GPU ---
Python Version: 3.13.15
OS: Linux 6.6.122+
GPU: Tesla T4
VRAM: 14.56 GB
Disk Space Available:
Disk Space Available: 53G
Step 1: Runtime and GPU check successful.

--- Step 2: Installing Ollama ---
Updating apt-get...
Installing zstd...
Step 2: Ollama installation successful.

--- Step 3: Starting Ollama Server ---
Waiting for Ollama server to respond...
Warning during server check: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=5)
Ollama server is active at http://localhost:11434
Step 3: Ollama server started successfully.

--- Step 4: Pulling and Verifying Models ---
Pulling qwen3:8b model...

Pulling qwen2.5-coder:7b model...

Pulling qwen3.5:latest model...

Local API Response: {
  "models": [
    {
      "name": "qwen2.5-coder:7b",
      "model": "qwen2.5-coder:7b",
      "modified_at": "2026-09-03T21:13:19.14714119Z",
      "size": 4683087561,
      "digest": "dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e7